# 01 · Heat vulnerability, air conditioning and population by neighborhood

**Objective.** Build the neighborhood layers of the map: the published Heat Vulnerability Index (HVI) rank,
the share of households without air conditioning, and the shares of residents who are Black (non-Hispanic)
and Hispanic (any race), for New York City's 262 2020 Neighborhood Tabulation Areas (NTAs), plus borough and
City Council district outlines.

| Inputs (`../inputs/`) | Source |
|---|---|
| `nta2020.geojson` | NYC Open Data 9nt8-h7nd, 2020 NTAs |
| `hvi_nta2020.csv` | NYC DOHMH Heat Vulnerability Index by 2020 NTA |
| `dcp_decennial_census_2020.xlsx` | NYC DCP Decennial Census core-geographies workbook (NTA2020 rows) |
| `boroughs_dcp_26b.geojson` | NYC DCP Borough Boundaries, release 26b |
| `council_districts.geojson` | NYC Open Data 872g-cjhh, City Council Districts |

Outputs go to `data/`: `neighborhoods.geojson`, `neighborhood_values.csv`, `boroughs.geojson`,
`council_districts.geojson`, `layers.json`. Full citations with retrieval dates are in
[`../inputs/README.md`](../inputs/README.md).

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()                      # run this notebook from its own directory (run_notebooks.py does)
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))
DATA = HERE / "data"
DATA.mkdir(exist_ok=True)

import geopandas as gpd
import numpy as np
import pandas as pd

from common import BLUE_RAMP, INPUTS, NO_DATA, RED_RAMP, SOURCES, load_hvi, write_geojson, write_json

NYC_POPULATION_2020 = 8804190  # Source: DCP 2020 Census workbook, NYC2020 row, Pop1


## 1. Join the HVI table to the neighborhood boundaries

The HVI table has one row per NTA, keyed by `NTACode`, with the rank (`HVI_RANK`, 1 lowest to 5 highest) and
the five factor columns the department builds the rank from: `SURFACE_TEMP`, `GREENSPACE`, `PCT_HOUSEHOLDS_AC`,
`MEDIAN_INCOME` and `PCT_BLACK_POP`. DOHMH describes the score as the sum of those factors assigned to quintiles
(NYC Open Data, Heat Vulnerability Index Rankings, 4mhf-duep; indicator 2411 description). This notebook uses the
rank and the air-conditioning column only. One NTA, BX0802
(Kingsbridge-Marble Hill), appears twice with identical values because Marble Hill is administratively in
Manhattan but physically in the Bronx; `load_hvi` checks that both rows carry the same rank and AC share before
dropping one, so the join is one-to-one.

DOHMH scores residential neighborhoods only: parks, airports, cemeteries and similar areas have no rank.

In [2]:
nta = gpd.read_file(INPUTS / "nta2020.geojson").to_crs(4326)
hvi = load_hvi()   # checks the duplicated BX0802 rows agree before dropping one
nta = nta.merge(hvi[["NTACode", "HVI_RANK", "PCT_HOUSEHOLDS_AC"]], left_on="nta2020", right_on="NTACode",
                how="left", validate="one_to_one")
nta = nta[["nta2020", "ntaname", "boroname", "HVI_RANK", "PCT_HOUSEHOLDS_AC", "geometry"]].rename(
    columns={"PCT_HOUSEHOLDS_AC": "pct_households_ac"})

assert len(nta) == 262 and nta.HVI_RANK.notna().sum() == 197
print(f"{len(nta)} neighborhoods, {nta.HVI_RANK.notna().sum()} with an HVI rank")
nta.HVI_RANK.value_counts(dropna=False).sort_index().rename("neighborhoods").to_frame().T


262 neighborhoods, 197 with an HVI rank


HVI_RANK,1.0,2.0,3.0,4.0,5.0,NaN
neighborhoods,37,40,40,40,40,65


## 2. Add 2020 Census counts and compute shares

From the DCP workbook's `NTA2020` rows: `Pop1` (total population), `BNH` (Black, non-Hispanic) and `Hsp1`
(Hispanic, any race). Shares are counts divided by total population, rounded to one decimal, and left null
where the population is zero. The NTA populations must sum to the city's 2020 population.

`pct_households_no_ac` is 100 minus the published households-with-AC percentage. It measures whether a
household has air conditioning, not whether the unit works or is used.

In [3]:
census = pd.read_excel(INPUTS / "dcp_decennial_census_2020.xlsx", sheet_name="2020")
counts = census.loc[census.GeoType == "NTA2020", ["GeoID", "Pop1", "BNH", "Hsp1"]].rename(
    columns={"Pop1": "population_2020", "BNH": "black_non_hispanic_count", "Hsp1": "hispanic_count"})
nta = nta.merge(counts, left_on="nta2020", right_on="GeoID", how="left", validate="one_to_one").drop(columns="GeoID")
assert nta.population_2020.notna().all() and nta.population_2020.sum() == NYC_POPULATION_2020

nta["pct_households_no_ac"] = 100 - nta.pct_households_ac
for field, count in [("pct_black_nh", "black_non_hispanic_count"), ("pct_hispanic", "hispanic_count")]:
    nta[field] = (nta[count] / nta.population_2020.replace(0, np.nan) * 100).round(1)

nta[["pct_households_no_ac", "pct_black_nh", "pct_hispanic"]].describe().round(1)


,pct_households_no_ac,pct_black_nh,pct_hispanic
count,197.0,246.0,246.0
mean,9.4,19.8,29.4
std,4.9,22.1,21.3
min,1.6,0.0,0.0
25%,5.9,2.8,12.6
50%,8.1,10.4,22.1
75%,12.2,29.4,40.8
max,24.2,84.5,100.0


## 3. Assign display colors

Each layer gets a five-step sequential ramp and a fixed set of breakpoints. HVI uses its five published ranks.
No AC uses 5-percentage-point steps (5, 10, 15, 20); the Census shares use 20-point steps (20, 40, 60, 80).
Bins are upper-inclusive: a value greater than a breakpoint enters the next bin. These are display intervals
chosen for legibility; they are not thresholds with any epidemiological meaning. Missing values are gray and
never mean zero.

Colors ship in the data so that classification and display cannot drift apart.

In [4]:
INTERVAL_RULE = "Upper bound inclusive: a value greater than a breakpoint enters the next bin."


def classify(values: pd.Series, cuts: list, ramp: list) -> pd.Series:
    return values.map(lambda v: NO_DATA if pd.isna(v) else ramp[sum(v > cut for cut in cuts)])


layer_specs = []
for key, title, field, cuts, unit, ramp, note, source in [
    ("hvi", "Heat vulnerability", "HVI_RANK", [1, 2, 3, 4], "score", RED_RAMP,
     "Published Heat Vulnerability Index rank per neighborhood, 1 (lowest) to 5 (highest).", SOURCES["hvi"]),
    ("no_ac", "No AC", "pct_households_no_ac", [5, 10, 15, 20], "%", RED_RAMP,
     "Households without air conditioning: 100 minus the published households-with-AC percentage. Measures availability, not whether a unit works or is used.", SOURCES["hvi"]),
    ("black", "Black population", "pct_black_nh", [20, 40, 60, 80], "%", BLUE_RAMP,
     "Black, non-Hispanic residents as a percentage of all residents, 2020 Census.", SOURCES["census"]),
    ("hispanic", "Hispanic population", "pct_hispanic", [20, 40, 60, 80], "%", BLUE_RAMP,
     "Hispanic residents of any race as a percentage of all residents, 2020 Census.", SOURCES["census"]),
]:
    nta[f"{key}_color"] = classify(nta[field], cuts, ramp)
    layer_specs.append({"id": key, "title": title, "value_field": field, "color_field": f"{key}_color", "unit": unit,
                        "breakpoints": cuts, "interval_rule": INTERVAL_RULE, "colors": ramp, "no_data_color": NO_DATA,
                        "note": note, "sources": [source]})

pd.DataFrame([{"layer": s["id"], "value field": s["value_field"], "breakpoints": s["breakpoints"], "ramp": s["colors"]} for s in layer_specs])


,layer,value field,breakpoints,ramp
0,hvi,HVI_RANK,"[1, 2, 3, 4]","[#f5ded8, #eeafa3, #e47c6c, #d84536, #c60101]"
1,no_ac,pct_households_no_ac,"[5, 10, 15, 20]","[#f5ded8, #eeafa3, #e47c6c, #d84536, #c60101]"
2,black,pct_black_nh,"[20, 40, 60, 80]","[#e5f3fb, #bedff4, #94ccee, #73bfe9, #53b1e3]"
3,hispanic,pct_hispanic,"[20, 40, 60, 80]","[#e5f3fb, #bedff4, #94ccee, #73bfe9, #53b1e3]"


## 4. Outline layers

Borough boundaries (water areas excluded) and City Council districts are used as outlines only. No
district-level statistics are computed: neighborhoods and districts do not nest, and the two boundary files
have different vintages and shorelines.

In [5]:
boroughs = gpd.read_file(INPUTS / "boroughs_dcp_26b.geojson").to_crs(4326)[["borocode", "boroname", "geometry"]]
boroughs["borocode"] = boroughs["borocode"].astype(str)
council = gpd.read_file(INPUTS / "council_districts.geojson").to_crs(4326)[["coundist", "geometry"]]
council["coundist"] = council["coundist"].astype(str)
assert len(boroughs) == 5 and len(council) == 51
boroughs[["borocode", "boroname"]]


,borocode,boroname
0,5,Staten Island
1,3,Brooklyn
2,4,Queens
3,1,Manhattan
4,2,Bronx


## 5. Write the outputs

`write_geojson` swaps each feature's coordinates for the simplified display polygon of the same ID from
[`../geometry/`](../geometry/README.md) and checks that every polygon is valid. Attribute values are the
ones computed above from the full-resolution sources.

In [6]:
write_geojson(DATA / "neighborhoods.geojson", nta, "nta2020", "neighborhoods",
              "2020 Neighborhood Tabulation Areas with the published Heat Vulnerability Index rank, household AC availability and 2020 Census composition. Null means unavailable or, for shares, zero population; never zero.",
              [SOURCES["nta"], SOURCES["hvi"], SOURCES["census"]])
nta.drop(columns="geometry").to_csv(DATA / "neighborhood_values.csv", index=False)
write_geojson(DATA / "boroughs.geojson", boroughs, "borocode", "boroughs", "Borough boundaries, water areas excluded.", [SOURCES["boroughs"]])
write_geojson(DATA / "council_districts.geojson", council, "coundist", "council",
              "City Council district boundaries, keyed by district number. No district-level statistics are computed.", [SOURCES["council"]])
write_json(DATA / "layers.json", {
    "description": "Map layers drawn from neighborhoods.geojson (value_field and color_field per layer), plus borough and council outlines. Bins are display intervals, not risk thresholds.",
    "sources": [SOURCES["boroughs"], SOURCES["nta"], SOURCES["hvi"], SOURCES["census"], SOURCES["council"]],
    "boroughs": {"file": "boroughs.geojson", "id_field": "borocode", "title": "Boroughs", "note": "Borough boundaries, water areas excluded."},
    "neighborhoods_file": "neighborhoods.geojson",
    "layers": layer_specs,
    "council": {"file": "council_districts.geojson", "id_field": "coundist", "title": "City Council districts",
                "stroke": "#474747", "stroke_width": 1.25, "stroke_dasharray": "5 3",
                "note": "District boundaries only. Neighborhood values are not district averages."},
})


wrote 01_heat_vulnerability_map/data/neighborhoods.geojson (262 features, 12,771 vertices)


wrote 01_heat_vulnerability_map/data/boroughs.geojson (5 features, 6,136 vertices)


wrote 01_heat_vulnerability_map/data/council_districts.geojson (51 features, 9,114 vertices)
wrote 01_heat_vulnerability_map/data/layers.json


## Results

Descriptive summaries of the layers just written. DOHMH builds the HVI from five inputs: surface temperature,
green space, air conditioning access, median income and the share of residents who are Black. So the No AC layer
and the HVI are not independent measures, and the rise in Black population share across ranks is partly built
into the index; Hispanic share is not an input. Composition by rank is a co-location, not a causal relationship.

In [7]:
scored = nta[nta.HVI_RANK.notna()]
by_rank = scored.groupby("HVI_RANK").agg(
    neighborhoods=("nta2020", "size"),
    population=("population_2020", "sum"),
    mean_pct_no_ac=("pct_households_no_ac", "mean"),
    black_nh=("black_non_hispanic_count", "sum"),
    hispanic=("hispanic_count", "sum"),
)
by_rank["pct_black_nh_popweighted"] = by_rank.black_nh / by_rank.population * 100
by_rank["pct_hispanic_popweighted"] = by_rank.hispanic / by_rank.population * 100
print(f"Households without AC: min {scored.pct_households_no_ac.min():.1f}%, median {scored.pct_households_no_ac.median():.1f}%, "
      f"max {scored.pct_households_no_ac.max():.1f}%; {int((scored.pct_households_no_ac > 20).sum())} neighborhoods above 20%")
by_rank.round(2).to_csv(DATA / "summary_by_hvi_rank.csv")   # the counts behind the population-weighted shares
by_rank.drop(columns=["black_nh", "hispanic"]).round(1)


Households without AC: min 1.6%, median 8.1%, max 24.2%; 8 neighborhoods above 20%


,neighborhoods,population,mean_pct_no_ac,pct_black_nh_popweighted,pct_hispanic_popweighted
HVI_RANK,,,,,
1.0,37,1576297,5.1,4.7,14.4
2.0,40,1790888,6.7,6.8,25.9
3.0,40,1743172,8.2,11.5,32.5
4.0,40,1898442,10.4,27.9,31.2
5.0,40,1787728,16.3,47.4,35.6
